# LLaMA-3-8B Sentiment Baseline – All-in-one Colab

Self-contained: all logic is in this notebook (no clone required).  
**Runtime → Change runtime type → GPU** before running.

---

**Sync with repo (Colab ↔ local)**

- **Use latest from repo in Colab:** In Colab: File → Upload notebook, and choose `colab/llama_sentiment_baseline_train.ipynb` from your repo. Or open from GitHub: open the repo, click this `.ipynb`, use the "Open in Colab" button if present.
- **Save Colab changes back to repo:** In Colab: File → Download → Download .ipynb. Replace `colab/llama_sentiment_baseline_train.ipynb` in your repo with the downloaded file.
- **Trained output:** After Section 7, download `output_final.zip`. Unzip and put the `final` folder contents into your repo at `llama_sentiment_baseline/output/final/` so `make test` works locally.


## 1. Install dependencies


In [1]:
!nvidia-smi
!pip install -q torch transformers peft datasets accelerate pandas scikit-learn textblob kagglehub "bitsandbytes>=0.46.1"
# For T4 16GB: after this cell, use Runtime → Restart session, then Run all. That makes 4-bit load correctly.
print("Done.")

/bin/bash: line 1: nvidia-smi: command not found
Done.


In [2]:
import os

os.environ["PYTORCH_ALLOC_CONF"] = (
    "expandable_segments:True"  # Before any torch/CUDA use
)
# Load Hugging Face token from Colab Secrets (Runtime → Secrets); add a secret named "hugging-face-token"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("hugging-face-token")
except Exception:
    pass  # Not in Colab or secret not set; set HF_TOKEN manually or login cell will prompt

## 2. Hugging Face login

**LLaMA is a gated model.** In Colab: add your token in **Runtime → Secrets** as **`hugging-face-token`** (the cell above loads it). Then run the cell below.

1. Open https://huggingface.co/meta-llama/Meta-Llama-3-8B and **request access** (accept the license).
2. Create a token at https://huggingface.co/settings/tokens and add it to Colab Secrets as **hugging-face-token**.


In [3]:
import os
from huggingface_hub import login

# Empty token causes "Illegal header value b'Bearer '". Use a real token from https://huggingface.co/settings/tokens
token = os.environ.get("HF_TOKEN", "").strip()
if token:
    login(token=token)
else:
    # Clear empty cached token so we get prompted again
    cache_token = os.path.expanduser("~/.cache/huggingface/token")
    if os.path.isfile(cache_token):
        with open(cache_token) as f:
            if not f.read().strip():
                os.remove(cache_token)
                print("Removed empty cached token.")
    login()  # will prompt; paste a valid token (not empty)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## 3. Download dataset (Kaggle)

Upload `kaggle.json` to Colab, or create `/root/.kaggle/kaggle.json` with your API key, then run below.


In [4]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)
# If you uploaded kaggle.json: uncomment and run
# !mv /content/kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json

import kagglehub

path = kagglehub.dataset_download("arhamrumi/amazon-product-reviews")
csvs = [f for f in os.listdir(path) if f.lower().endswith(".csv")]
DATASET_PATH = os.path.join(path, csvs[0]) if csvs else None
print("Dataset path:", DATASET_PATH)

Using Colab cache for faster access to the 'amazon-product-reviews' dataset.
Dataset path: /kaggle/input/amazon-product-reviews/Reviews.csv


## 4. Config and all training logic (paper-aligned)

**Paper Section 2.1–2.2 (itmconf_dai2024_04021):** Data pipeline order: **TextBlob DQC** → **stratified sampling** (equal per rating) → **VGST** (1%, optional; set `use_vgst=True` for full alignment, slower) → **oversample neutral** → cap at **max_samples** (Table 1: 2000). Prompts: directive, one-shot, CoT.

**Is this paper-exact?** Matches: Table 1 (LR 5e-5, epochs 3, batch 3, grad_accum 4, max_samples 2000, cutoff 1024, LoRA r=4/alpha=64, trainable 3 layers, NEFTune 0.02, val 0.2, cosine LR), Section 2.1–2.2 (TextBlob, stratified, **VGST 1%**, oversample neutral, one-shot, CoT). We read the full CSV (max_read=None) and use VGST by default. If Colab is slow or OOMs: set max_read=50000, or data_cfg.use_vgst=False, or per_device_train_batch_size=2.


In [5]:
import os, random
from dataclasses import dataclass
from typing import Optional, Any

import torch
import pandas as pd
from datasets import Dataset


@dataclass
class ModelConfig:
    model_name_or_path: str = "meta-llama/Meta-Llama-3-8B"
    use_fast_tokenizer: bool = True
    trust_remote_code: bool = True
    lora_r: int = 4
    lora_alpha: int = 64
    lora_dropout: float = 0.0
    lora_target_modules: list = None
    trainable_layers: int = 3
    use_rslora: bool = False
    use_dora: bool = False


@dataclass
class TrainingConfig:
    output_dir: str = "./output"
    overwrite_output_dir: bool = True
    num_train_epochs: int = 3
    learning_rate: float = 5e-5
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    per_device_train_batch_size: int = 3
    per_device_eval_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    max_seq_length: int = 1024
    fp16: bool = True
    bf16: bool = False
    optim: str = "adamw_torch"
    lr_scheduler_type: str = "cosine"
    warmup_steps: int = 0
    logging_steps: int = 5
    save_steps: int = 100
    save_total_limit: int = 2
    evaluation_strategy: str = "steps"
    eval_steps: int = 100
    val_ratio: float = 0.2
    max_samples: Optional[int] = 2000
    neftune_alpha: float = 0.02
    enable_external_logger: bool = True
    seed: int = 42


@dataclass
class DataConfig:
    text_column: str = "review_text"
    label_column: str = "rating"
    negative_labels: tuple = (1, 2)
    neutral_labels: tuple = (3,)
    positive_labels: tuple = (4, 5)
    use_textblob_filter: bool = True
    use_stratified_sampling: bool = True
    stratified_max_total: Optional[int] = None
    use_vgst: bool = True
    vgst_target_ratio: float = 0.01
    vgst_batch_size: int = 32
    vgst_wishlist_len: int = 10
    oversample_neutral: bool = True
    neutral_oversample_ratio: float = 2.0
    use_one_shot: bool = True
    use_cot: bool = True
    cot_phrase: str = "Let's take it one step at a time."


SENTIMENT_INSTRUCTION = "Evaluate the sentiment expressed in user reviews and classify each one according to its sentiment rating. Use a five-point scale: 1-2 negative, 3 neutral, 4-5 positive."
RATING_DESCRIPTIONS = {
    1: "Comments show a high level of dissatisfaction and negativity (rating 1).",
    2: "Although still negative, the user's sentiment may be slightly softened (rating 2).",
    3: "The review is generally neutral, with both positive and negative aspects (rating 3).",
    4: "Users have a positive experience overall, with room for improvement (rating 4).",
    5: "High level of satisfaction and positive emotions (rating 5).",
}

TEXT_COLS = ("review_text", "text", "Text", "review", "Review", "content", "body")
RATING_COLS = ("rating", "Rating", "score", "Score", "overall", "rating_star")

In [6]:
def _infer_columns(cols):
    t = next((c for c in TEXT_COLS if c in cols), None)
    r = next((c for c in RATING_COLS if c in cols), None)
    return t, r


def load_raw_data(path, text_column, label_column, max_rows=None):
    ext = os.path.splitext(path)[1].lower()
    rows = []
    if ext == ".csv":
        df = pd.read_csv(path, nrows=max_rows)
        infer_t, infer_r = _infer_columns(df.columns.tolist())
        use_t = text_column if text_column in df.columns else (infer_t or "review_text")
        use_r = label_column if label_column in df.columns else (infer_r or "rating")
        for _, r in df.iterrows():
            text = (
                str(r[use_t])
                if use_t in r
                else str(r.get("review_text", r.get("review", "")))
            )
            try:
                rating = int(float(r[use_r]))
            except (ValueError, TypeError):
                rating = 3
            rating = max(1, min(5, rating))
            rows.append({"text": text, "rating": rating})
    else:
        raise ValueError(f"Unsupported: {path}")
    return rows


def filter_textblob(rows):
    try:
        from textblob import TextBlob
    except ImportError:
        return rows
    kept = []
    for r in rows:
        text, rating = r.get("text", ""), r.get("rating", 3)
        try:
            pol = float(TextBlob(text).sentiment.polarity)
        except:
            kept.append(r)
            continue
        if pol < 0 and rating > 3:
            continue
        if pol > 0 and rating < 3:
            continue
        kept.append(r)
    return kept


def stratified_sample(rows, samples_per_rating=None, max_total=None, rng=None):
    rng = rng or random.Random()
    by_rating = {i: [] for i in range(1, 6)}
    for row in rows:
        if row.get("rating") in by_rating:
            by_rating[row["rating"]].append(row)
    if max_total and samples_per_rating is None:
        samples_per_rating = max(1, max_total // 5)
    if samples_per_rating is None:
        counts = [len(by_rating[i]) for i in range(1, 6)]
        samples_per_rating = min(counts) if counts else 0
    out = []
    for i in range(1, 6):
        pool = list(by_rating[i])
        rng.shuffle(pool)
        out.extend(pool[:samples_per_rating])
    rng.shuffle(out)
    return out


def oversample_neutral(rows, neutral_rating=3, multiplier=2.0, rng=None):
    rng = rng or random.Random()
    neutral = [r for r in rows if r.get("rating") == neutral_rating]
    others = [r for r in rows if r.get("rating") != neutral_rating]
    n_extra = max(0, int(len(neutral) * (multiplier - 1.0)))
    extra = rng.choices(neutral, k=n_extra)
    out = others + neutral + extra
    rng.shuffle(out)
    return out


def apply_paper_preprocessing(
    rows, data_cfg, tokenizer=None, max_samples=None, seed=42
):
    rng = random.Random(seed)
    if data_cfg.use_textblob_filter:
        rows = filter_textblob(rows)
    if data_cfg.use_stratified_sampling:
        rows = stratified_sample(
            rows, max_total=data_cfg.stratified_max_total or max_samples, rng=rng
        )
    if data_cfg.oversample_neutral:
        rows = oversample_neutral(
            rows, multiplier=data_cfg.neutral_oversample_ratio, rng=rng
        )
    if max_samples and len(rows) > max_samples:
        rng.shuffle(rows)
        rows = rows[:max_samples]
    return rows

In [7]:
def format_one_shot(review, rating, cfg):
    desc = RATING_DESCRIPTIONS.get(rating, f"Rating {rating}.")
    return f"Review: {review}\nSentiment (1-5): {rating}. {desc}"


def build_prompt(review, cfg, one_shot_example=None):
    parts = [SENTIMENT_INSTRUCTION]
    if cfg.use_cot:
        parts.append(cfg.cot_phrase)
    if one_shot_example and cfg.use_one_shot:
        parts.append("\n\nExample:\n" + one_shot_example)
    parts.append("\n\nReview to classify:\n" + review)
    parts.append("\nSentiment (1-5):")
    return "\n".join(parts)


def prepare_conversation(text, rating, cfg, one_shot=None):
    prompt = build_prompt(text, cfg, one_shot_example=one_shot)
    return {"prompt": prompt, "answer": str(rating), "rating": rating, "text": text}


def create_one_shot_pool(samples, cfg, pool_size=5):
    by_rating = {r: [] for r in range(1, 6)}
    for s in samples:
        r = s.get("rating")
        if r in by_rating:
            by_rating[r].append({"text": s.get("text", ""), "rating": int(r)})
    pool = []
    for r in range(1, 6):
        if by_rating[r]:
            pool.append(random.choice(by_rating[r]))
    if not pool and samples:
        pool = [
            {
                "text": samples[0].get("text", ""),
                "rating": int(samples[0].get("rating", 3)),
            }
        ]
    return pool[:pool_size]


def get_one_shot_from_pool(pool, cfg, rng):
    if not pool or not cfg.use_one_shot:
        return None
    ex = rng.choice(pool)
    return format_one_shot(ex["text"], ex["rating"], cfg)

In [ ]:
def build_sft_dataset(data_path, tokenizer, data_cfg, training_cfg, seed=42):
    rng = random.Random(seed)
    if not data_path or not os.path.isfile(data_path):
        rows = [
            {"text": "This product is great.", "rating": 5},
            {"text": "Terrible.", "rating": 1},
            {"text": "Okay.", "rating": 3},
        ] * 10
    else:
        # None = read full CSV (paper 500k; needs more RAM). 50000 = faster, less RAM.
        max_read = None
        rows = load_raw_data(
            data_path, data_cfg.text_column, data_cfg.label_column, max_rows=max_read
        )
        rows = apply_paper_preprocessing(
            rows,
            data_cfg,
            tokenizer=tokenizer,
            max_samples=training_cfg.max_samples,
            seed=seed,
        )
    if training_cfg.max_samples and len(rows) > training_cfg.max_samples:
        rng.shuffle(rows)
        rows = rows[: training_cfg.max_samples]
    rng.shuffle(rows)
    n_val = max(1, int(len(rows) * training_cfg.val_ratio))
    val_rows, train_rows = rows[:n_val], rows[n_val:]
    one_shot_pool = create_one_shot_pool(train_rows, data_cfg, pool_size=5)
    max_len, pad_id = (
        training_cfg.max_seq_length,
        tokenizer.pad_token_id or tokenizer.eos_token_id,
    )

    def format_and_tokenize(examples, split):
        prompts, answers = [], []
        for ex in examples:
            one_shot = (
                get_one_shot_from_pool(one_shot_pool, data_cfg, rng)
                if split == "train"
                else None
            )
            f = prepare_conversation(ex["text"], ex["rating"], data_cfg, one_shot)
            prompts.append(f["prompt"])
            answers.append(f["answer"])
        input_ids_list, attention_mask_list, labels_list = [], [], []
        for p, a in zip(prompts, answers):
            full = p + " " + a
            tok = tokenizer(
                full,
                truncation=True,
                max_length=max_len,
                padding="max_length",
                return_tensors=None,
            )
            prompt_tok = tokenizer(
                p, truncation=True, max_length=max_len, return_tensors=None
            )
            prompt_len = len(prompt_tok["input_ids"])
            input_ids = list(tok["input_ids"])
            labels = [-100] * prompt_len + input_ids[prompt_len:]
            while len(input_ids) < max_len:
                input_ids.append(pad_id)
                labels.append(-100)
            input_ids, labels = input_ids[:max_len], labels[:max_len]
            attention_mask = [1] * len(input_ids) + [0] * (max_len - len(input_ids))
            attention_mask = attention_mask[:max_len]
            input_ids_list.append(input_ids)
            attention_mask_list.append(attention_mask)
            labels_list.append(labels)
        return {
            "input_ids": input_ids_list,
            "attention_mask": attention_mask_list,
            "labels": labels_list,
        }

    train_processed = format_and_tokenize(train_rows, "train")
    val_processed = format_and_tokenize(val_rows, "val")
    return Dataset.from_dict(train_processed), Dataset.from_dict(val_processed)


print("Data/preprocessing helpers defined.")

Data/preprocessing helpers defined.


: 

## 5. Load model and tokenizer (LoRA, 4-bit)


In [9]:
!pip install -U "bitsandbytes>=0.46.1" -q
_bnb_error = None
try:
    import bitsandbytes  # Only need this package; transformers uses it via BitsAndBytesConfig
    _bnb_available = True
except Exception as e:
    _bnb_available = False
    _bnb_error = e

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model


def get_lora_config(cfg):
    num_layers = 32
    trainable = min(cfg.trainable_layers, num_layers)
    layers_to_transform = (
        list(range(num_layers - trainable, num_layers))
        if trainable < num_layers
        else None
    )
    return LoraConfig(
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=cfg.lora_target_modules or ["q_proj", "v_proj"],
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        inference_mode=False,
        use_rslora=cfg.use_rslora,
        use_dora=cfg.use_dora,
        layers_to_transform=layers_to_transform,
        layers_pattern="layers",
    )


def freeze_base(model):
    for name, param in model.named_parameters():
        if "lora" not in name.lower():
            param.requires_grad = False


model_cfg = ModelConfig()
use_4bit = _bnb_available
if not use_4bit:
    print("4-bit skipped (bitsandbytes not loadable).")
    if _bnb_error is not None:
        print("Error:", _bnb_error)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
    if gpu_mem_gb < 18:
        raise RuntimeError(
            "LLaMA-3-8B in fp16 needs ~16GB; your GPU has {:.1f}GB so training will OOM. "
            "You must use 4-bit: Runtime → Restart session, then run from cell 1 (Run all). "
            "That loads bitsandbytes before the model.".format(gpu_mem_gb)
        )
    print("Using fp16 (GPU has >= 18GB).")
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_cfg.model_name_or_path,
    use_fast=model_cfg.use_fast_tokenizer,
    trust_remote_code=model_cfg.trust_remote_code,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Loading model (4-bit)..." if use_4bit else "Loading model (fp16)...")
model_kwargs = {
    "trust_remote_code": model_cfg.trust_remote_code,
    "device_map": "auto",
    "torch_dtype": torch.float16,
}
if use_4bit:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )
model = AutoModelForCausalLM.from_pretrained(
    model_cfg.model_name_or_path, **model_kwargs
)
print("Applying LoRA...")
model = get_peft_model(model, get_lora_config(model_cfg))
freeze_base(model)
model.print_trainable_parameters()
print("Model ready.")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading model (4-bit)...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

: 

: 

## 6. Build dataset and train


In [ ]:
DATASET_PATH = globals().get("DATASET_PATH", None)
# Paper-exact (itmconf_dai2024_04021 Table 1 + §2.1–2.2): batch 3, grad_accum 4, max_seq 1024, max_samples 2000.
training_cfg = TrainingConfig(
    output_dir="./output",
    per_device_train_batch_size=3,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    max_seq_length=1024,
    max_samples=2000,
)
data_cfg = DataConfig()
data_cfg.stratified_max_total = training_cfg.max_samples

torch.manual_seed(training_cfg.seed)
print("Building dataset...")
train_ds, eval_ds = build_sft_dataset(
    DATASET_PATH, tokenizer, data_cfg, training_cfg, seed=training_cfg.seed
)
print(f"Train: {len(train_ds)}, Eval: {len(eval_ds)}")

Building dataset...
Train: 1600, Eval: 400


In [ ]:
from transformers import Trainer, TrainingArguments
from transformers.data.data_collator import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    max_length=training_cfg.max_seq_length,
    pad_to_multiple_of=8 if training_cfg.fp16 else None,
    label_pad_token_id=-100,
    return_tensors="pt",
)
training_args = TrainingArguments(
    output_dir=training_cfg.output_dir,
    num_train_epochs=training_cfg.num_train_epochs,
    per_device_train_batch_size=training_cfg.per_device_train_batch_size,
    per_device_eval_batch_size=training_cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=training_cfg.gradient_accumulation_steps,
    learning_rate=training_cfg.learning_rate,
    weight_decay=training_cfg.weight_decay,
    max_grad_norm=training_cfg.max_grad_norm,
    lr_scheduler_type=training_cfg.lr_scheduler_type,
    warmup_steps=training_cfg.warmup_steps,
    logging_steps=training_cfg.logging_steps,
    save_steps=training_cfg.save_steps,
    save_total_limit=training_cfg.save_total_limit,
    eval_strategy=training_cfg.evaluation_strategy,
    eval_steps=training_cfg.eval_steps,
    fp16=training_cfg.fp16,
    bf16=training_cfg.bf16,
    optim=training_cfg.optim,
    seed=training_cfg.seed,
    report_to="tensorboard" if training_cfg.enable_external_logger else "none",
    neftune_noise_alpha=training_cfg.neftune_alpha,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
)
torch.cuda.empty_cache()
print("Training...")
trainer.train()
trainer.save_model(os.path.join(training_cfg.output_dir, "final"))
tokenizer.save_pretrained(os.path.join(training_cfg.output_dir, "final"))
print("Done.")

Training...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2402: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Step,Training Loss,Validation Loss
100,0.006649,0.004961
200,0.004497,0.003541
300,0.002872,0.002660
400,0.002613,0.002509


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2402: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2402: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2402: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2402: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Done.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 7. Download adapter

**VS Code + Colab:** Direct download often doesn't work. Run the cell below; it will copy the zip to **Google Drive** (you may be asked to mount Drive once). Then open [drive.google.com](https://drive.google.com) → folder **Colab_Output** → download **output_final.zip**.


In [ ]:
!zip -r output_final.zip output/final
import os
import shutil

def try_download():
    try:
        from google.colab import files
        files.download("output_final.zip")
        print("Download started. Save the file when your browser prompts.")
        return True
    except Exception:
        return False

def copy_to_drive():
    drive_dir = "/content/drive/MyDrive"
    if not os.path.isdir(drive_dir):
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except Exception as e:
            return False, str(e)
    out_dir = os.path.join(drive_dir, "Colab_Output")
    os.makedirs(out_dir, exist_ok=True)
    dest = os.path.join(out_dir, "output_final.zip")
    shutil.copy2("output_final.zip", dest)
    return True, dest

if not try_download():
    ok, path = copy_to_drive()
    if ok:
        print("Saved to Google Drive:", path)
        print("Open https://drive.google.com → Colab_Output → download output_final.zip")
    else:
        print("Direct download not available (e.g. VS Code Colab).")
        print("Option 1: Mount Drive above and re-run this cell to copy to Drive.")
        print("Option 2: File is at /content/output_final.zip on the runtime.")
        print("         In VS Code: Colab panel → check for Files / Download or right-click to save.")

updating: output/final/ (stored 0%)
updating: output/final/adapter_config.json (deflated 56%)
updating: output/final/README.md (deflated 66%)
updating: output/final/tokenizer.json (deflated 85%)
updating: output/final/tokenizer_config.json (deflated 46%)
updating: output/final/training_args.bin (deflated 53%)
updating: output/final/adapter_model.safetensors (deflated 8%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started. Save the file when your browser prompts.


## 8. Test / Evaluate

Run evaluation: **Table 3** (Accuracy, Precision, Recall, F1) and **Table 2** (BLEU-4, ROUGE-1, samples/sec). Uses the same model and tokenizer from training.


In [ ]:
!pip install -q rouge-score sacrebleu
import re
import time
from sklearn.metrics import precision_recall_fscore_support
import sacrebleu
from rouge_score import rouge_scorer


def extract_rating_from_output(text):
    """Parse model output to get a single digit 1-5."""
    text = text.strip()
    match = re.search(r"[1-5](?:\s|$|\.)", text)
    if match:
        return int(match.group()[0])
    digits = re.findall(r"\d", text)
    for d in reversed(digits):
        if d in "12345":
            return int(d)
    return 3


MAX_EVAL_SAMPLES = 500
model.eval()

if not DATASET_PATH or not os.path.isfile(DATASET_PATH):
    print("No DATASET_PATH; skipping evaluation.")
else:
    rows = load_raw_data(
        DATASET_PATH, data_cfg.text_column, data_cfg.label_column, max_rows=10000
    )
    rng = random.Random(42)
    rng.shuffle(rows)
    examples = rows[:MAX_EVAL_SAMPLES]
    print(f"Evaluating on {len(examples)} examples...")
    preds, refs = [], []
    decoded_texts = []
    ref_texts = []
    t0 = time.perf_counter()
    for ex in examples:
        prompt = build_prompt(ex["text"], data_cfg, one_shot_example=None)
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=training_cfg.max_seq_length,
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=16,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )
        decoded = tokenizer.decode(
            out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
        )
        decoded_texts.append(decoded.strip() or " ")
        ref_texts.append(str(ex["rating"]))
        preds.append(extract_rating_from_output(decoded))
        refs.append(ex["rating"])
    elapsed = time.perf_counter() - t0
    samples_per_sec = len(examples) / elapsed if elapsed > 0 else 0.0
    correct = sum(1 for p, r in zip(preds, refs) if p == r)
    accuracy = correct / len(refs)
    prec, rec, f1, _ = precision_recall_fscore_support(
        refs, preds, average="weighted", zero_division=0
    )
    bleu = sacrebleu.corpus_bleu(decoded_texts, [ref_texts]).score
    scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=False)
    rouge1_scores = [scorer.score(ref_texts[i], decoded_texts[i])["rouge1"].fmeasure for i in range(len(ref_texts))]
    rouge1 = sum(rouge1_scores) / len(rouge1_scores) * 100.0 if rouge1_scores else 0.0
    print("")
    print("=== Results (all paper metrics) ===")
    print("Table 3 – Classification:")
    print(f"  Accuracy:   {accuracy:.4f} ({correct}/{len(refs)})")
    print(f"  Precision:  {prec:.4f}")
    print(f"  Recall:     {rec:.4f}")
    print(f"  F1:         {f1:.4f}")
    print("Table 2 – Generation & throughput:")
    print(f"  BLEU-4:     {bleu:.4f}")
    print(f"  ROUGE-1:    {rouge1:.2f}")
    print(f"  Samples/s:  {samples_per_sec:.3f}")